# 承接关系 encoder 对比学习训练

用 `training/build_dataset.py` 产出的 `(text_a, text_b, label)` JSONL，做 SimCSE 式 in-batch 对比学习，微调 sentence-transformer 模型。

**步骤**：① 安装依赖 → ② 上传数据 → ③ 训练 → ④ 保存到 Drive

> 建议开启「代码执行程序 → 更改运行时类型 → T4 GPU」。

In [ ]:
# ① 安装依赖
!pip install -q sentence-transformers datasets accelerate

In [ ]:
# ② 上传训练数据 JSONL（本地 build_dataset.py 产出的 pairs_*.jsonl）
from google.colab import files
uploaded = files.upload()
data_path = [f for f in uploaded if f.endswith('.jsonl')][0]
print('data:', data_path)

In [ ]:
# ③ 训练
import json
from datasets import Dataset
from sentence_transformers import (
    SentenceTransformer, SentenceTransformerTrainer, SentenceTransformerTrainingArguments,
)
from sentence_transformers.sentence_transformer.losses import MultipleNegativesRankingLoss

# ===== 配置（按需改）=====
MODEL = "BAAI/bge-small-en-v1.5"   # 或 BAAI/bge-m3（更强但更慢、显存更多）
EPOCHS = 3
BATCH_SIZE = 32                    # T4 16G：bge-small 可用 64，bge-m3 用 16~32
LR = 2e-5
SCALE = 20.0                       # MNRL 温度系数
OUTPUT_DIR = "/content/model"
# ========================

pairs = []
with open(data_path, encoding='utf-8') as f:
    for line in f:
        r = json.loads(line)
        if r.get('label') == 1 and r.get('text_a') and r.get('text_b'):
            pairs.append({'anchor': r['text_a'], 'positive': r['text_b']})
print('正样本对数量:', len(pairs))

model = SentenceTransformer(MODEL)
dataset = Dataset.from_list(pairs)
loss = MultipleNegativesRankingLoss(model, scale=SCALE)

args = SentenceTransformerTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=LR,
    fp16=True,               # T4 混合精度加速
    bf16=False,
    logging_steps=100,
    save_strategy='epoch',
    report_to=[],            # 关闭 wandb
    dataloader_drop_last=True,
)

trainer = SentenceTransformerTrainer(model=model, args=args, train_dataset=dataset, loss=loss)
trainer.train()
model.save_pretrained(OUTPUT_DIR)
print('saved ->', OUTPUT_DIR)

In [ ]:
# ④ 保存到 Google Drive（可选）
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/longersession_model
!cp -r /content/model /content/drive/MyDrive/longersession_model/
print('已保存到 Drive')

## 训练后如何使用

训练好的模型是 sentence-transformers 格式，可直接加载：

```python
from sentence_transformers import SentenceTransformer
m = SentenceTransformer('/content/model')
v = m.encode(['上一轮的文本', '下一轮的文本'], normalize_embeddings=True)
cos = float(v[0] @ v[1])   # 承接分数，越高越连贯
```

也可以把 `/content/model` 目录拷回本地，替换 `src/longersession/encoder/bge_m3.py` 里的模型路径，或直接当 encoder 接入切割管线。